In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor
from sklearn.svm import SVC
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam
from keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load dataset
df = pd.read_csv("River_Plastic_Waste_Risk_Scenarios_2015_2060.csv")

# Define 2015 features for regression
features_2015 = [
    'Population_2015', 'Waste_Generation_2015_tons', 'Plastic_Waste_2015_tons',
    'Mismanaged_Waste_2015_tons', 'Plastic_to_River_2015_tons', 'River_Length_km',
    'Basin_Area_km2', 'Urbanization_2015_pct', 'GDP_per_capita_2015',
    'Policy_Strength_2015', 'Waste_Collection_Rate_2015'
]
target = 'Plastic_Waste_2060_tons'

X = df[features_2015]
y = df[target]

# Train-test split for regression
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Models ---
results = {}

# Linear Regression
lr = LinearRegression().fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
results['LinearRegression'] = {
    'MAE': mean_absolute_error(y_test, y_pred_lr),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_lr)),
    'R2': r2_score(y_test, y_pred_lr)
}

# Random Forest
rf = RandomForestRegressor(random_state=42).fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results['RandomForest'] = {
    'MAE': mean_absolute_error(y_test, y_pred_rf),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    'R2': r2_score(y_test, y_pred_rf)
}

# XGBoost
xgb = XGBRegressor(random_state=42).fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
results['XGBoost'] = {
    'MAE': mean_absolute_error(y_test, y_pred_xgb),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
    'R2': r2_score(y_test, y_pred_xgb)
}

# MLP Regressor (Keras)
mlp = Sequential([
    Dense(64, input_shape=(X_train_scaled.shape[1],), activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])
mlp.compile(optimizer=Adam(0.001), loss='mean_squared_error')
mlp.fit(X_train_scaled, y_train, epochs=50, batch_size=32, verbose=0)
y_pred_mlp = mlp.predict(X_test_scaled).flatten()
results['MLP'] = {
    'MAE': mean_absolute_error(y_test, y_pred_mlp),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_mlp)),
    'R2': r2_score(y_test, y_pred_mlp)
}

# Save predictions to new CSV
df_pred = df.copy()
df_pred['Predicted_Plastic_Waste_2060'] = rf.predict(X)  # Using RF model as final
df_pred.to_csv('river_plastic_predictions.csv', index=False)


/Users/yo/anaconda3/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 873us/step


In [3]:
# Classify Risk Score Change into categories: Low, Medium, High
def classify_risk(score):
    if score < 0.33:
        return 'Low'
    elif score < 0.66:
        return 'Medium'
    else:
        return 'High'

df['Risk_Category'] = df['Risk_Score_Change'].apply(classify_risk)

# Features and target for classification
X_cls = df[features_2015]
y_cls = df['Risk_Category']

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_cls)

# Train-test split
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(X_cls, y_encoded, test_size=0.2, random_state=42)

# Standardize features
X_cls_train_scaled = scaler.fit_transform(X_cls_train)
X_cls_test_scaled = scaler.transform(X_cls_test)

# --- Classification Models ---
classification_results = {}

# Logistic Regression
logreg = LogisticRegression(max_iter=1000).fit(X_cls_train_scaled, y_cls_train)
y_pred_logreg = logreg.predict(X_cls_test_scaled)
classification_results['LogisticRegression'] = classification_report(y_cls_test, y_pred_logreg, output_dict=True)

# SVM
svm = SVC().fit(X_cls_train_scaled, y_cls_train)
y_pred_svm = svm.predict(X_cls_test_scaled)
classification_results['SVM'] = classification_report(y_cls_test, y_pred_svm, output_dict=True)

# Random Forest Classifier
rfc = RandomForestClassifier(random_state=42).fit(X_cls_train, y_cls_train)
y_pred_rfc = rfc.predict(X_cls_test)
classification_results['RandomForest'] = classification_report(y_cls_test, y_pred_rfc, output_dict=True)

# Add predicted risk categories to DataFrame
df['Predicted_Risk_Category'] = label_encoder.inverse_transform(rfc.predict(X_cls))

# Save results
df.to_csv('river_risk_classification.csv', index=False)